In [29]:
import pandas as pd
import numpy as np
import os
from getpass import getpass

from sqlalchemy import create_engine
from sqlalchemy import text

from sqlalchemy.types import (
    VARCHAR,
    Integer,
    Float,
    DateTime,
    Text
)

In [30]:
DATA_DIR = r"D:\Portfolio\SQL\Revenue-Leakage-SQL-Project\data\raw"

DB_USER = "root"
DB_PASSWORD = getpass("Enter MySQL Password: ")

DB_HOST = "localhost"
DB_PORT = 3306

DB_NAME = "revenue_leakage_db"

Enter MySQL Password:  ········


In [31]:
server_engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}"
)

with server_engine.connect() as conn:

    conn.execute(
        text(f"""
        DROP DATABASE IF EXISTS {DB_NAME}
        """)
    )

    conn.execute(
        text(f"""
        CREATE DATABASE {DB_NAME}
        CHARACTER SET utf8mb4
        COLLATE utf8mb4_unicode_ci
        """)
    )

    conn.commit()

print("Database Created Successfully")

Database Created Successfully


In [32]:
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected Successfully")

Connected Successfully


In [33]:
customers = pd.read_csv(
    os.path.join(DATA_DIR, "olist_customers_dataset.csv")
)

orders = pd.read_csv(
    os.path.join(DATA_DIR, "olist_orders_dataset.csv")
)

order_items = pd.read_csv(
    os.path.join(DATA_DIR, "olist_order_items_dataset.csv")
)

payments = pd.read_csv(
    os.path.join(DATA_DIR, "olist_order_payments_dataset.csv")
)

reviews = pd.read_csv(
    os.path.join(DATA_DIR, "olist_order_reviews_dataset.csv")
)

products = pd.read_csv(
    os.path.join(DATA_DIR, "olist_products_dataset.csv")
)

sellers = pd.read_csv(
    os.path.join(DATA_DIR, "olist_sellers_dataset.csv")
)

geolocation = pd.read_csv(
    os.path.join(DATA_DIR, "olist_geolocation_dataset.csv")
)

product_category_translation = pd.read_csv(
    os.path.join(DATA_DIR,
                 "product_category_name_translation.csv")
)

print("All Files Loaded")

All Files Loaded


In [34]:
orders_dates = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in orders_dates:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

reviews_dates = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in reviews_dates:
    reviews[col] = pd.to_datetime(
        reviews[col],
        errors="coerce"
    )

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

print("Dates Converted")

Dates Converted


In [35]:
customer_dtype = {
    "customer_id": VARCHAR(50),
    "customer_unique_id": VARCHAR(50),
    "customer_zip_code_prefix": Integer(),
    "customer_city": VARCHAR(100),
    "customer_state": VARCHAR(5)
}

orders_dtype = {
    "order_id": VARCHAR(50),
    "customer_id": VARCHAR(50),
    "order_status": VARCHAR(30),
    "order_purchase_timestamp": DateTime(),
    "order_approved_at": DateTime(),
    "order_delivered_carrier_date": DateTime(),
    "order_delivered_customer_date": DateTime(),
    "order_estimated_delivery_date": DateTime()
}

order_items_dtype = {
    "order_id": VARCHAR(50),
    "order_item_id": Integer(),
    "product_id": VARCHAR(50),
    "seller_id": VARCHAR(50),
    "shipping_limit_date": DateTime(),
    "price": Float(),
    "freight_value": Float()
}

payments_dtype = {
    "order_id": VARCHAR(50),
    "payment_sequential": Integer(),
    "payment_type": VARCHAR(30),
    "payment_installments": Integer(),
    "payment_value": Float()
}

reviews_dtype = {
    "review_id": VARCHAR(50),
    "order_id": VARCHAR(50),
    "review_score": Integer(),
    "review_comment_title": Text(),
    "review_comment_message": Text(),
    "review_creation_date": DateTime(),
    "review_answer_timestamp": DateTime()
}

products_dtype = {
    "product_id": VARCHAR(50),
    "product_category_name": VARCHAR(100),
    "product_name_lenght": Float(),
    "product_description_lenght": Float(),
    "product_photos_qty": Float(),
    "product_weight_g": Float(),
    "product_length_cm": Float(),
    "product_height_cm": Float(),
    "product_width_cm": Float()
}

sellers_dtype = {
    "seller_id": VARCHAR(50),
    "seller_zip_code_prefix": Integer(),
    "seller_city": VARCHAR(100),
    "seller_state": VARCHAR(5)
}

In [36]:
customers.to_sql(
    "customers",
    engine,
    if_exists="replace",
    index=False,
    dtype=customer_dtype
)

orders.to_sql(
    "orders",
    engine,
    if_exists="replace",
    index=False,
    dtype=orders_dtype
)

order_items.to_sql(
    "order_items",
    engine,
    if_exists="replace",
    index=False,
    dtype=order_items_dtype
)

payments.to_sql(
    "payments",
    engine,
    if_exists="replace",
    index=False,
    dtype=payments_dtype
)

reviews.to_sql(
    "reviews",
    engine,
    if_exists="replace",
    index=False,
    dtype=reviews_dtype
)

products.to_sql(
    "products",
    engine,
    if_exists="replace",
    index=False,
    dtype=products_dtype
)

sellers.to_sql(
    "sellers",
    engine,
    if_exists="replace",
    index=False,
    dtype=sellers_dtype
)

geolocation.to_sql(
    "geolocation",
    engine,
    if_exists="replace",
    index=False
)

product_category_translation.to_sql(
    "product_category_translation",
    engine,
    if_exists="replace",
    index=False
)

print("All Tables Uploaded")

All Tables Uploaded


In [37]:
with engine.connect() as conn:

    conn.execute(text("""
        ALTER TABLE customers
        ADD PRIMARY KEY(customer_id)
    """))

    conn.execute(text("""
        ALTER TABLE orders
        ADD PRIMARY KEY(order_id)
    """))

    conn.execute(text("""
        ALTER TABLE sellers
        ADD PRIMARY KEY(seller_id)
    """))

    conn.execute(text("""
        ALTER TABLE products
        ADD PRIMARY KEY(product_id)
    """))

    conn.commit()

print("Primary Keys Added")

Primary Keys Added


In [38]:
tables = [
    "customers",
    "orders",
    "order_items",
    "payments",
    "reviews",
    "products",
    "sellers",
    "geolocation",
    "product_category_translation"
]

validation = []

for table in tables:

    query = f"""
    SELECT COUNT(*) AS row_count
    FROM {table}
    """

    count = pd.read_sql(query, engine)

    validation.append({
        "table_name": table,
        "row_count": int(count.iloc[0,0])
    })

validation_df = pd.DataFrame(validation)

display(validation_df)

,table_name,row_count
0,customers,99441
1,orders,99441
2,order_items,112650
3,payments,103886
4,reviews,99224
5,products,32951
6,sellers,3095
7,geolocation,1000163
8,product_category_translation,71


In [40]:
for table in [
    "customers",
    "orders",
    "order_items",
    "products",
    "sellers"
]:

    print("\n")
    print("="*60)
    print(table.upper())
    print("="*60)

    display(
        pd.read_sql(
            f"DESCRIBE {table}",
            engine
        )
    )




CUSTOMERS


,Field,Type,Null,Key,Default,Extra
0,customer_id,varchar(50),NO,PRI,None,
1,customer_unique_id,varchar(50),YES,,None,
2,customer_zip_code_prefix,int,YES,,None,
3,customer_city,varchar(100),YES,,None,
4,customer_state,varchar(5),YES,,None,




ORDERS


,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(50),NO,PRI,None,
1,customer_id,varchar(50),YES,,None,
2,order_status,varchar(30),YES,,None,
3,order_purchase_timestamp,datetime,YES,,None,
4,order_approved_at,datetime,YES,,None,
5,order_delivered_carrier_date,datetime,YES,,None,
6,order_delivered_customer_date,datetime,YES,,None,
7,order_estimated_delivery_date,datetime,YES,,None,




ORDER_ITEMS


,Field,Type,Null,Key,Default,Extra
0,order_id,varchar(50),YES,,None,
1,order_item_id,int,YES,,None,
2,product_id,varchar(50),YES,,None,
3,seller_id,varchar(50),YES,,None,
4,shipping_limit_date,datetime,YES,,None,
5,price,float,YES,,None,
6,freight_value,float,YES,,None,




PRODUCTS


,Field,Type,Null,Key,Default,Extra
0,product_id,varchar(50),NO,PRI,None,
1,product_category_name,varchar(100),YES,,None,
2,product_name_lenght,float,YES,,None,
3,product_description_lenght,float,YES,,None,
4,product_photos_qty,float,YES,,None,
5,product_weight_g,float,YES,,None,
6,product_length_cm,float,YES,,None,
7,product_height_cm,float,YES,,None,
8,product_width_cm,float,YES,,None,




SELLERS


,Field,Type,Null,Key,Default,Extra
0,seller_id,varchar(50),NO,PRI,None,
1,seller_zip_code_prefix,int,YES,,None,
2,seller_city,varchar(100),YES,,None,
3,seller_state,varchar(5),YES,,None,
